In [184]:
# EDA
import pandas as pd
import plotly.express as px

# ML
from sklearn.datasets import load_iris
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# HP
import optuna

### Carga de Dados

In [185]:
iris = load_iris()

In [186]:
# Transforma iris em um DataFrame
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)

In [187]:
df_iris['target'] = iris.target

In [188]:
df_iris.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


In [189]:
df_iris.head(20)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
5,5.4,3.9,1.7,0.4,0
6,4.6,3.4,1.4,0.3,0
7,5.0,3.4,1.5,0.2,0
8,4.4,2.9,1.4,0.2,0
9,4.9,3.1,1.5,0.1,0


In [190]:
df_iris.tail(20)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
130,7.4,2.8,6.1,1.9,2
131,7.9,3.8,6.4,2.0,2
132,6.4,2.8,5.6,2.2,2
133,6.3,2.8,5.1,1.5,2
134,6.1,2.6,5.6,1.4,2
135,7.7,3.0,6.1,2.3,2
136,6.3,3.4,5.6,2.4,2
137,6.4,3.1,5.5,1.8,2
138,6.0,3.0,4.8,1.8,2
139,6.9,3.1,5.4,2.1,2


### EDA

In [191]:
df_iris.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


In [192]:
df_iris.shape

(150, 5)

In [193]:
df_iris.target.value_counts()

target
0    50
1    50
2    50
Name: count, dtype: int64

In [194]:
df_iris.target.value_counts(normalize=True)

target
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64

In [195]:
X = df_iris.drop('target', axis=1)
y = df_iris['target']

In [196]:
scaler = StandardScaler()
X_tranformed = scaler.fit_transform(X)

In [197]:
def gmm_objective(trial):
    n_components = trial.suggest_int('n_components', 2, 10)
    covariance_type = trial.suggest_categorical('covariance_type', ['full', 'tied', 'diag', 'spherical'])

    gmm = GaussianMixture(n_components=n_components, covariance_type=covariance_type, random_state=42)

    gmm.fit(X_tranformed)

    bic_gmm = gmm.bic(X_tranformed)

    return bic_gmm

In [198]:
search_space = {'n_components': [3, 4, 5, 6, 7, 8, 9, 10],
                'covariance_type': ['full', 'tied', 'diag', 'spherical']}

sampler = optuna.samplers.GridSampler(search_space=search_space)

estudo_gmm = optuna.create_study(direction='minimize', sampler=sampler)

[I 2026-06-06 15:02:38,338] A new study created in memory with name: no-name-64398997-4837-4908-abb7-f8dc0f462b16


In [199]:
estudo_gmm.optimize(gmm_objective, n_trials=32)

[I 2026-06-06 15:02:38,352] Trial 0 finished with value: 860.0529735540067 and parameters: {'n_components': 6, 'covariance_type': 'tied'}. Best is trial 0 with value: 860.0529735540067.
[I 2026-06-06 15:02:38,359] Trial 1 finished with value: 967.5949202389686 and parameters: {'n_components': 9, 'covariance_type': 'diag'}. Best is trial 0 with value: 860.0529735540067.
[I 2026-06-06 15:02:38,372] Trial 2 finished with value: 850.7362324512615 and parameters: {'n_components': 5, 'covariance_type': 'tied'}. Best is trial 2 with value: 850.7362324512615.
[I 2026-06-06 15:02:38,393] Trial 3 finished with value: 901.5377319262867 and parameters: {'n_components': 5, 'covariance_type': 'full'}. Best is trial 2 with value: 850.7362324512615.
[I 2026-06-06 15:02:38,397] Trial 4 finished with value: 1033.1586626992714 and parameters: {'n_components': 3, 'covariance_type': 'diag'}. Best is trial 2 with value: 850.7362324512615.
[I 2026-06-06 15:02:38,405] Trial 5 finished with value: 865.90952937

In [200]:
best_params = estudo_gmm.best_params

In [201]:
best_gmm = GaussianMixture(n_components=best_params['n_components'], covariance_type=best_params['covariance_type'], random_state=42)

best_gmm.fit(X_tranformed)

best_bic = best_gmm.bic(X_tranformed)

In [202]:
print("Quantidade ideal de componentes: ", best_params['n_components'])
print("Tipo de Covariância: ", best_params['covariance_type'])
print("BIC do melhor modelo: ", best_bic)

Quantidade ideal de componentes:  3
Tipo de Covariância:  full
BIC do melhor modelo:  841.1905492967948


In [203]:
clusters_gmm = best_gmm.predict(X_tranformed)

In [204]:
clusters_gmm

array([1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 2, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [205]:
clusters_gmm_prob = best_gmm.predict_proba(X_tranformed)

In [206]:
clusters_gmm_prob


array([[2.74503381e-011, 1.00000000e+000, 0.00000000e+000],
       [1.16033052e-007, 9.99999884e-001, 0.00000000e+000],
       [9.72921434e-009, 9.99999990e-001, 5.88168010e-278],
       [2.02193473e-007, 9.99999798e-001, 3.33173699e-141],
       [9.67775891e-012, 1.00000000e+000, 0.00000000e+000],
       [1.14329874e-012, 1.00000000e+000, 0.00000000e+000],
       [5.99406192e-009, 9.99999994e-001, 4.07722398e-049],
       [4.78561583e-010, 1.00000000e+000, 0.00000000e+000],
       [4.58916917e-007, 1.19519047e-001, 8.80480494e-001],
       [9.28895274e-008, 9.99999907e-001, 0.00000000e+000],
       [1.09664613e-012, 1.00000000e+000, 0.00000000e+000],
       [3.80456745e-009, 9.99999996e-001, 0.00000000e+000],
       [1.96191178e-007, 9.99999804e-001, 0.00000000e+000],
       [8.18276018e-008, 3.32205264e-002, 9.66779392e-001],
       [1.24741926e-016, 1.00000000e+000, 0.00000000e+000],
       [1.33001229e-018, 1.00000000e+000, 0.00000000e+000],
       [1.50274710e-014, 1.00000000e+000

In [207]:
df_iris['cluster'] = clusters_gmm.astype(int)

In [208]:
df_iris.head(10)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,cluster
0,5.1,3.5,1.4,0.2,0,1
1,4.9,3.0,1.4,0.2,0,1
2,4.7,3.2,1.3,0.2,0,1
3,4.6,3.1,1.5,0.2,0,1
4,5.0,3.6,1.4,0.2,0,1
5,5.4,3.9,1.7,0.4,0,1
6,4.6,3.4,1.4,0.3,0,1
7,5.0,3.4,1.5,0.2,0,1
8,4.4,2.9,1.4,0.2,0,2
9,4.9,3.1,1.5,0.1,0,1


In [209]:
px.scatter(df_iris, x='sepal width (cm)', y='sepal length (cm)', color='cluster')

In [210]:
px.scatter(df_iris, x='petal width (cm)', y='petal length (cm)', color='cluster')

In [211]:
px.scatter(df_iris, x='petal width (cm)', y='sepal length (cm)', color='cluster')

In [212]:
px.scatter(df_iris, x='sepal width (cm)', y='petal length (cm)', color='cluster')